# Advanced 07 lab — Robustness, uncertainty, and failure recovery

**Scenario.** A three-class visual inspection system was validated at Site A. Site B is development-only: it selects calibration, OOD, conformal, selective-risk, and recovery policy. Site C is reporting-only. We must detect unsupported operating conditions, quantify imperfect reliability evidence, make bounded decisions, and verify recovery.

**Evidence boundary.** This credential-free notebook uses procedural 16×16 images, a tiny local CNN ensemble, and simulated recovery. It is **not a foundation-model benchmark, production reliability result, physical safety case, or deployment authorization**. Model scores are untrusted measurements. Application code owns required evidence, policy, attempts, terminal states, and verification.

![A stressed observation moves through capability and uncertainty evidence, a trusted risk policy, and verified recovery.](assets/reliability-lifecycle.svg)


## 1. Reproducible environment and optional-tool governance

The default path uses common PyTorch, NumPy, pandas, scikit-learn, and Matplotlib APIs. Optional reliability frameworks remain disabled and revision pinned. No `lab.py`, remote code, weights, services, credentials, or private data are used.


In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import platform
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Callable, Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, roc_auc_score, roc_curve

SEED = 20260920
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)

DEVICE = torch.device("cpu")
ARTIFACT_DIR = Path.cwd() / ".artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

CV_ENABLE_TORCH_UNCERTAINTY = False
CV_ENABLE_OPENOOD = False
CV_ENABLE_ALIBI_DETECT = False
CV_ENABLE_MAPIE = False
CV_ENABLE_ROBUSTBENCH = False

OPTIONAL_TOOL_MANIFESTS = {
    "torch_uncertainty": {"enabled": CV_ENABLE_TORCH_UNCERTAINTY, "revision": "3f82fe5d15a7bf877a821731baadef1e4731c31d"},
    "openood": {"enabled": CV_ENABLE_OPENOOD, "revision": "3c35632ee91b54b09d1f085d04f94744cece7d0b"},
    "alibi_detect": {"enabled": CV_ENABLE_ALIBI_DETECT, "revision": "c2fd0e05c648d353467bdb15fd6149a103b3a981"},
    "mapie": {"enabled": CV_ENABLE_MAPIE, "revision": "7e888f5249bf942912207d398525d3259a5f07c0"},
    "robustbench": {"enabled": CV_ENABLE_ROBUSTBENCH, "revision": "review_exact_commit_before_use"},
}
assert not any(item["enabled"] for item in OPTIONAL_TOOL_MANIFESTS.values())

environment = {
    "python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__,
    "pandas": pd.__version__, "sklearn": sklearn.__version__, "device": str(DEVICE),
    "seed": SEED, "credential_free": True,
}
environment


## 2. Typed contracts before inference

Metric direction, positive class, slice, population, and aggregation are part of the evidence contract. Sensor validation and recovery state are application-owned. Missing required evidence never becomes acceptance.


In [ ]:
@dataclass(frozen=True)
class SourceContract:
    source: str
    role: Literal["training", "development_only", "reporting_only_no_changes"]
    camera_revision: str


@dataclass(frozen=True)
class MetricSpec:
    name: str
    direction: Literal["higher_is_better", "lower_is_better"]
    positive_class: str
    population: str
    unit: str
    aggregation: str


@dataclass(frozen=True)
class ScoreSpec:
    name: str
    orientation: Literal["higher_is_more_ood", "lower_is_more_ood"]
    normalized_name: str
    normalization: Literal["identity", "one_minus"]
    positive_class: Literal["OOD"] = "OOD"


@dataclass(frozen=True)
class SensorObservation:
    observation_id: str
    source: str
    timestamp_s: float
    image_present: bool
    modality_agreement: Literal["agree", "contradict", "unknown"]


@dataclass(frozen=True)
class ReliabilityPolicy:
    version: str
    selected_on: str
    temperature: float
    ood_score_name: str
    ood_threshold: float
    accept_confidence_min: float
    minimum_required_coverage: float
    conformal_alpha: float
    conformal_quantile: float
    max_reobservations: int
    max_alternate_attempts: int
    freshness_max_s: float
    costs: dict[str, float]
    digest: str


@dataclass(frozen=True)
class RiskDecision:
    operation_id: str
    terminal_state: Literal["accepted", "verified_recovery", "human_review", "unresolved", "blocked_invalid_input"]
    reason: str
    attempts: int
    recovery_attempt: str | None
    candidate_result: dict[str, Any] | None
    verification_source: str | None
    verified_success: bool
    authorization: Literal["none"] = "none"


@dataclass(frozen=True)
class RecoveryCandidate:
    operation_id: str
    attempt_id: str
    method: Literal["primary", "reobservation", "alternate_centroid"]
    prediction: int
    decision_support: float


@dataclass(frozen=True)
class RecoveryProposal:
    operation_id: str
    terminal_state: Literal["accepted", "verification_required", "human_review", "unresolved", "blocked_invalid_input"]
    reason: str
    attempts: int
    candidate: RecoveryCandidate | None


@dataclass(frozen=True)
class VerificationReceipt:
    attempt_id: str
    verification_source: Literal["synthetic_evaluation_oracle"]
    verified_success: bool


METRIC_SPECS = {
    "accuracy": MetricSpec("accuracy", "higher_is_better", "correct", "named source/slice", "fraction", "macro over examples"),
    "ece": MetricSpec("ece", "lower_is_better", "not applicable", "named source/slice", "probability gap", "confidence-bin weighted"),
    "ood_auroc": MetricSpec("ood_auroc", "higher_is_better", "OOD", "ID plus named OOD set", "area", "pairwise ranking"),
    "selective_risk": MetricSpec("selective_risk", "lower_is_better", "error", "accepted examples", "error fraction", "conditional on acceptance"),
    "verified_recovery_rate": MetricSpec("verified_recovery_rate", "higher_is_better", "verified success", "attempted recoveries", "fraction", "attempt-weighted"),
}

SCORE_SPECS = {
    "msp": ScoreSpec("msp", "lower_is_more_ood", "msp_ood", "one_minus"),
    "entropy": ScoreSpec("entropy", "higher_is_more_ood", "entropy", "identity"),
    "energy": ScoreSpec("energy", "higher_is_more_ood", "energy", "identity"),
    "centroid_distance": ScoreSpec("centroid_distance", "higher_is_more_ood", "centroid_distance", "identity"),
    "mahalanobis": ScoreSpec("mahalanobis", "higher_is_more_ood", "mahalanobis", "identity"),
}


def canonical_hash(value: Any) -> str:
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), default=str).encode()
    return hashlib.sha256(payload).hexdigest()


SOURCES = (
    SourceContract("Site A", "training", "camera-a-v1"),
    SourceContract("Site B", "development_only", "camera-b-v1"),
    SourceContract("Site C", "reporting_only_no_changes", "camera-c-v1"),
)
assert SOURCES[2].role == "reporting_only_no_changes"


## 3. Procedural Site A/B/C visual data

Three classes use different spatial structures rather than label-coded vectors. Site B adds mild camera translation and brightness variation. Site C adds a stronger reporting-only transform. Synthetic data isolates mechanics; it does not establish real inspection quality.


In [ ]:
CLASS_NAMES = ("vertical_seal", "horizontal_seal", "diagonal_seal")


def class_template(class_id: int, size: int = 16) -> torch.Tensor:
    image = torch.zeros(1, size, size)
    if class_id == 0:
        image[:, 3:13, 7:9] = 1.0
    elif class_id == 1:
        image[:, 7:9, 3:13] = 1.0
    else:
        for i in range(3, 13):
            image[:, i, i] = 1.0
            if i + 1 < size:
                image[:, i, i + 1] = 0.8
    return image


def make_site(source: str, role: str, n_per_class: int, seed: int) -> tuple[torch.Tensor, torch.Tensor]:
    generator = torch.Generator().manual_seed(seed)
    images, labels = [], []
    for class_id in range(len(CLASS_NAMES)):
        for _ in range(n_per_class):
            image = class_template(class_id)
            image = image + 0.07 * torch.randn(image.shape, generator=generator)
            if source == "Site B":
                image = torch.roll(image, shifts=(0, 1), dims=(1, 2)) * 0.90 + 0.05
            elif source == "Site C":
                image = torch.roll(image, shifts=(1, -1), dims=(1, 2)) * 0.72 + 0.16
                image[:, 2:5, 11:14] *= 0.25
            images.append(image.clamp(0, 1))
            labels.append(class_id)
    order = torch.randperm(len(labels), generator=generator)
    return torch.stack(images)[order], torch.tensor(labels)[order]


site_a_train = make_site("Site A", "training", 70, SEED + 1)
site_a_eval = make_site("Site A", "legacy", 35, SEED + 2)
site_b_dev = make_site("Site B", "development_only", 35, SEED + 3)
site_c_report = make_site("Site C", "reporting_only_no_changes", 35, SEED + 4)

split_manifest = pd.DataFrame([
    {"source": "Site A train", "role": "training", "rows": len(site_a_train[1])},
    {"source": "Site A eval", "role": "legacy_reporting", "rows": len(site_a_eval[1])},
    {"source": "Site B", "role": "development_only", "rows": len(site_b_dev[1])},
    {"source": "Site C", "role": "reporting_only_no_changes", "rows": len(site_c_report[1])},
])
split_manifest


## 4. Train a tiny CNN ensemble and freeze the base

Three independently initialized models make disagreement observable. Dropout supports a separate MC-dropout proxy. The ensemble cost is measured rather than assumed free.


In [ ]:
class TinyReliabilityCNN(nn.Module):
    def __init__(self, seed: int):
        super().__init__()
        torch.manual_seed(seed)
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 12, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.embedding = nn.Linear(12 * 4 * 4, 16)
        self.dropout = nn.Dropout(p=0.22)
        self.head = nn.Linear(16, len(CLASS_NAMES))

    def forward(self, x: torch.Tensor, return_embedding: bool = False):
        hidden = self.features(x).flatten(1)
        embedding = torch.tanh(self.embedding(hidden))
        logits = self.head(self.dropout(embedding))
        return (logits, embedding) if return_embedding else logits


def train_model(seed: int, epochs: int = 75) -> tuple[TinyReliabilityCNN, list[float]]:
    model = TinyReliabilityCNN(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.018)
    history = []
    x, y = site_a_train
    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach()))
    model.eval()
    return model, history


training_start = time.perf_counter()
ensemble = [train_model(SEED + offset)[0] for offset in (10, 20, 30)]
ensemble_train_time_s = time.perf_counter() - training_start


@torch.no_grad()
def ensemble_outputs(x: torch.Tensor) -> dict[str, torch.Tensor]:
    member_logits, member_embeddings = [], []
    for model in ensemble:
        model.eval()
        logits, embedding = model(x, return_embedding=True)
        member_logits.append(logits)
        member_embeddings.append(embedding)
    logits = torch.stack(member_logits)
    probabilities = logits.softmax(dim=-1)
    return {
        "member_logits": logits,
        "mean_logits": logits.mean(0),
        "member_probabilities": probabilities,
        "probabilities": probabilities.mean(0),
        "embedding": torch.stack(member_embeddings).mean(0),
    }


def state_hash(models: list[nn.Module]) -> str:
    values = [tensor.detach().cpu().tolist() for model in models for tensor in model.state_dict().values()]
    return canonical_hash(values)


BASE_ENSEMBLE_HASH = state_hash(ensemble)
assert len(BASE_ENSEMBLE_HASH) == 64
{"ensemble_members": len(ensemble), "training_time_s": ensemble_train_time_s, "base_hash": BASE_ENSEMBLE_HASH[:12]}


## 5. Capability and calibration metrics from first principles

ECE is implemented manually so bin population and direction stay visible. Brier and NLL complement it. Each result names the evaluated source.


In [ ]:
def expected_calibration_error(probabilities: torch.Tensor, labels: torch.Tensor, bins: int = 10) -> tuple[float, pd.DataFrame]:
    confidence, prediction = probabilities.max(dim=1)
    correct = prediction.eq(labels).float()
    rows, ece = [], 0.0
    boundaries = torch.linspace(0, 1, bins + 1)
    for index in range(bins):
        lower, upper = boundaries[index], boundaries[index + 1]
        mask = (confidence > lower) & (confidence <= upper) if index else (confidence >= lower) & (confidence <= upper)
        count = int(mask.sum())
        if count:
            accuracy = float(correct[mask].mean())
            mean_confidence = float(confidence[mask].mean())
            weight = count / len(labels)
            gap = abs(accuracy - mean_confidence)
            ece += weight * gap
            rows.append({"bin": index, "count": count, "accuracy": accuracy, "confidence": mean_confidence, "gap": gap})
    return float(ece), pd.DataFrame(rows)


def brier_score(probabilities: torch.Tensor, labels: torch.Tensor) -> float:
    target = F.one_hot(labels, num_classes=probabilities.shape[1]).float()
    return float((probabilities - target).pow(2).sum(dim=1).mean())


def negative_log_likelihood(probabilities: torch.Tensor, labels: torch.Tensor) -> float:
    return float(-torch.log(probabilities[torch.arange(len(labels)), labels].clamp_min(1e-9)).mean())


def capability_report(source: str, probabilities: torch.Tensor, labels: torch.Tensor) -> dict[str, float | str]:
    prediction = probabilities.argmax(1)
    ece, _ = expected_calibration_error(probabilities, labels)
    return {
        "source": source,
        "accuracy": accuracy_score(labels.numpy(), prediction.numpy()),
        "macro_f1": f1_score(labels.numpy(), prediction.numpy(), average="macro"),
        "ece": ece,
        "brier": brier_score(probabilities, labels),
        "nll": negative_log_likelihood(probabilities, labels),
        "mean_confidence": float(probabilities.max(1).values.mean()),
    }


base_reports = []
for name, data in (("Site A", site_a_eval), ("Site B", site_b_dev), ("Site C", site_c_report)):
    base_reports.append(capability_report(name, ensemble_outputs(data[0])["probabilities"], data[1]))
base_metrics = pd.DataFrame(base_reports)
base_metrics


## 6. Six corruption families × five severities

Every transform is deterministic for a given seed and retains the label contract. Compression is a quantization proxy—not a codec benchmark. Results remain disaggregated by family and severity.


In [ ]:
def corrupt(images: torch.Tensor, family: str, severity: int, seed: int) -> torch.Tensor:
    assert 1 <= severity <= 5
    x = images.clone()
    if family == "gaussian_noise":
        generator = torch.Generator().manual_seed(seed)
        x = x + (0.06 * severity) * torch.randn(x.shape, generator=generator)
    elif family == "blur":
        kernel = 1 + 2 * min(3, severity)
        x = F.avg_pool2d(x, kernel_size=kernel, stride=1, padding=kernel // 2)
    elif family == "brightness":
        x = x + 0.10 * severity
    elif family == "contrast":
        mean = x.mean(dim=(2, 3), keepdim=True)
        x = mean + (x - mean) * max(0.15, 1 - 0.17 * severity)
    elif family == "compression_proxy":
        levels = max(3, 36 - 6 * severity)
        x = torch.round(x * (levels - 1)) / (levels - 1)
    elif family == "occlusion":
        block = 2 + 2 * severity
        start = (16 - block) // 2
        x[:, :, start:start + block, start:start + block] = 0.5
    else:
        raise ValueError(f"unknown_corruption:{family}")
    return x.clamp(0, 1)


CORRUPTIONS = ("gaussian_noise", "blur", "brightness", "contrast", "compression_proxy", "occlusion")
corruption_rows = []
clean_accuracy = float(base_metrics.set_index("source").loc["Site A", "accuracy"])
for family in CORRUPTIONS:
    for severity in range(1, 6):
        x = corrupt(site_a_eval[0], family, severity, SEED + severity)
        probabilities = ensemble_outputs(x)["probabilities"]
        report = capability_report(f"{family}:{severity}", probabilities, site_a_eval[1])
        corruption_rows.append({"corruption": family, "severity": severity, **report})
corruption_results = pd.DataFrame(corruption_rows)

degradation_summary = []
for family, group in corruption_results.groupby("corruption"):
    ordered = group.sort_values("severity")
    severities = np.concatenate([[0], ordered["severity"].to_numpy()])
    degradation = np.concatenate([[0.0], clean_accuracy - ordered["accuracy"].to_numpy()])
    degradation_summary.append({
        "corruption": family,
        "course_local_degradation_auc": float(np.trapezoid(degradation, severities)),
        "worst_accuracy": float(ordered["accuracy"].min()),
        "worst_severity": int(ordered.loc[ordered["accuracy"].idxmin(), "severity"]),
    })
degradation_summary = pd.DataFrame(degradation_summary).sort_values("worst_accuracy")
assert len(corruption_results) == 30
degradation_summary


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for family, group in corruption_results.groupby("corruption"):
    ax.plot(group["severity"], group["accuracy"], marker="o", label=family)
ax.axhline(clean_accuracy, color="#16324F", linestyle="--", label="clean")
ax.set(xlabel="severity", ylabel="accuracy", title="Corruption degradation curves")
ax.set_ylim(0, 1.03)
ax.legend(ncol=2, fontsize=8)
plt.show()


## 7. Ambiguity, near OOD, far OOD, and operational shift

Ambiguous inputs blend two valid classes. Near OOD uses a novel plus-shaped structure that shares strokes with known classes. Far OOD is unrelated random texture. Site C is labelled operational shift. These categories are intentionally separate.


In [ ]:
def make_novel_ood(n: int, seed: int) -> torch.Tensor:
    generator = torch.Generator().manual_seed(seed)
    images = []
    for _ in range(n):
        # A plus-shaped near-OOD class shares strokes with two known classes.
        image = 0.72 * class_template(0) + 0.72 * class_template(1)
        image += 0.06 * torch.randn(image.shape, generator=generator)
        images.append(image.clamp(0, 1))
    return torch.stack(images)


def make_far_ood(n: int, seed: int) -> torch.Tensor:
    generator = torch.Generator().manual_seed(seed)
    return torch.rand((n, 1, 16, 16), generator=generator)


ambiguous_labels = site_b_dev[1][:45]
ambiguous_weights = torch.linspace(0.50, 0.80, len(ambiguous_labels))
ambiguous_images = torch.stack([
    weight * image + (1 - weight) * class_template((int(label) + 1) % len(CLASS_NAMES))
    for image, label, weight in zip(site_b_dev[0][:45], ambiguous_labels, ambiguous_weights)
])
near_ood_dev = make_novel_ood(70, SEED + 50)
far_ood_dev = make_far_ood(70, SEED + 51)
near_ood_report = make_novel_ood(70, SEED + 52)
far_ood_report = make_far_ood(70, SEED + 53)
unsupported_clean_images = torch.roll(site_b_dev[0], shifts=(2, -2), dims=(2, 3))
unsupported_labels = site_b_dev[1]
corrupted_unsupported_images = corrupt(unsupported_clean_images, "occlusion", 4, SEED + 54)

slice_manifest = pd.DataFrame([
    {"slice": "clean ID", "role": "capability", "rows": len(site_a_eval[1])},
    {"slice": "ambiguous ID", "role": "uncertainty stress", "rows": len(ambiguous_images)},
    {"slice": "near OOD", "role": "development stress", "rows": len(near_ood_dev)},
    {"slice": "far OOD", "role": "development stress", "rows": len(far_ood_dev)},
    {"slice": "clean unsupported", "role": "diagnostic development stress", "rows": len(unsupported_clean_images)},
    {"slice": "corrupted unsupported", "role": "diagnostic development stress", "rows": len(corrupted_unsupported_images)},
    {"slice": "Site C", "role": "reporting_only_no_changes", "rows": len(site_c_report[1])},
])
slice_manifest


## 8. MSP, entropy, margin, ensemble disagreement, and MC dropout

All scores are recorded with orientation. High `uncertainty_score` means more caution. MC dropout is a stochastic proxy, not a calibrated posterior.


In [ ]:
def predictive_scores(outputs: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    probabilities = outputs["probabilities"]
    top2 = probabilities.topk(2, dim=1).values
    entropy = -(probabilities * probabilities.clamp_min(1e-9).log()).sum(1)
    vote = outputs["member_probabilities"].argmax(2)
    vote_disagreement = 1 - torch.stack([(vote == class_id).float().mean(0) for class_id in range(len(CLASS_NAMES))]).max(0).values
    predictive_variance = outputs["member_probabilities"].var(0, unbiased=False).mean(1)
    return {
        "msp": probabilities.max(1).values,
        "entropy": entropy,
        "margin": top2[:, 0] - top2[:, 1],
        "vote_disagreement": vote_disagreement,
        "predictive_variance": predictive_variance,
    }


@torch.no_grad()
def mc_dropout_scores(model: TinyReliabilityCNN, x: torch.Tensor, passes: int = 20) -> torch.Tensor:
    model.train()
    probabilities = torch.stack([model(x).softmax(1) for _ in range(passes)])
    model.eval()
    return probabilities.var(0, unbiased=False).mean(1)


uncertainty_rows = []
for name, images in (("clean", site_a_eval[0]), ("ambiguous", ambiguous_images), ("near_ood", near_ood_dev), ("far_ood", far_ood_dev), ("site_c", site_c_report[0])):
    outputs = ensemble_outputs(images)
    scores = predictive_scores(outputs)
    dropout = mc_dropout_scores(ensemble[0], images)
    uncertainty_rows.append({
        "slice": name,
        "mean_msp": float(scores["msp"].mean()),
        "mean_entropy": float(scores["entropy"].mean()),
        "mean_margin": float(scores["margin"].mean()),
        "mean_vote_disagreement": float(scores["vote_disagreement"].mean()),
        "mean_predictive_variance": float(scores["predictive_variance"].mean()),
        "mean_mc_dropout_variance": float(dropout.mean()),
    })
uncertainty_comparison = pd.DataFrame(uncertainty_rows)
uncertainty_comparison


## 9. Signature failure: wrong and confident

A severe stress pool is searched for actual model errors, then ranked by MSP. The result shows that a normalized distribution can be concentrated and wrong.


In [ ]:
stress_images = torch.cat([
    corrupt(site_a_eval[0], "blur", 5, SEED + 80),
    corrupt(site_a_eval[0], "occlusion", 5, SEED + 81),
    corrupt(site_a_eval[0], "contrast", 5, SEED + 82),
    torch.stack([class_template((int(label) + 1) % 3) for label in site_a_eval[1]]),
])
stress_labels = site_a_eval[1].repeat(4)
stress_probabilities = ensemble_outputs(stress_images)["probabilities"]
stress_prediction = stress_probabilities.argmax(1)
wrong_mask = stress_prediction.ne(stress_labels)
if not wrong_mask.any():
    # Deterministic semantic-conflict stress: observation resembles the next class
    # while the source contract retains the original label. It teaches that
    # confidence cannot resolve contradictory evidence.
    stress_images = torch.stack([class_template((int(label) + 1) % 3) for label in site_a_eval[1]])
    stress_labels = site_a_eval[1]
    stress_probabilities = ensemble_outputs(stress_images)["probabilities"]
    stress_prediction = stress_probabilities.argmax(1)
    wrong_mask = stress_prediction.ne(stress_labels)

wrong_indices = torch.where(wrong_mask)[0]
wrong_confidence = stress_probabilities[wrong_indices].max(1).values
selected_wrong = wrong_indices[wrong_confidence.argmax()]
false_confidence_example = {
    "true_label": CLASS_NAMES[int(stress_labels[selected_wrong])],
    "prediction": CLASS_NAMES[int(stress_prediction[selected_wrong])],
    "max_softmax_probability": float(stress_probabilities[selected_wrong].max()),
    "predictive_entropy": float(-(stress_probabilities[selected_wrong] * stress_probabilities[selected_wrong].clamp_min(1e-9).log()).sum()),
    "warning": "wrong + concentrated softmax; confidence is not support evidence",
}
assert false_confidence_example["prediction"] != false_confidence_example["true_label"]
false_confidence_example


## 10. Reliability diagrams, ECE, Brier, and NLL

The clean and shifted diagrams can differ even with the same model. Calibration must name its population.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
reliability_tables = {}
for axis, (name, data) in zip(axes, (("Site B", site_b_dev), ("Site C", site_c_report))):
    probabilities = ensemble_outputs(data[0])["probabilities"]
    ece, table = expected_calibration_error(probabilities, data[1])
    reliability_tables[name] = table
    axis.plot([0, 1], [0, 1], "--", color="#52606D")
    axis.plot(table["confidence"], table["accuracy"], marker="o", color="#2F6BFF")
    axis.set(xlabel="mean confidence", ylabel="empirical accuracy", title=f"{name} · ECE={ece:.3f}", xlim=(0, 1), ylim=(0, 1))
plt.tight_layout()
plt.show()


## 11. Temperature scaling on Site B; frozen report on Site C

The grid search minimizes Site-B NLL. It never reads Site-C labels until after the policy hash is frozen. Positive temperature preserves every argmax.


In [ ]:
def probabilities_at_temperature(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    assert temperature > 0
    return (logits / temperature).softmax(1)


site_b_outputs = ensemble_outputs(site_b_dev[0])
temperatures = np.geomspace(0.35, 4.0, 80)
temperature_search = pd.DataFrame([
    {"temperature": float(t), "nll": negative_log_likelihood(probabilities_at_temperature(site_b_outputs["mean_logits"], float(t)), site_b_dev[1])}
    for t in temperatures
])
SELECTED_TEMPERATURE = float(temperature_search.loc[temperature_search["nll"].idxmin(), "temperature"])
temperature_policy_fragment = {"selected_on": "Site B development only", "temperature": SELECTED_TEMPERATURE, "objective": "minimum NLL"}
temperature_hash_before_site_c = canonical_hash(temperature_policy_fragment)

calibration_rows = []
for source, data in (("Site B", site_b_dev), ("Site C", site_c_report)):
    outputs = ensemble_outputs(data[0])
    raw = outputs["probabilities"]
    calibrated = probabilities_at_temperature(outputs["mean_logits"], SELECTED_TEMPERATURE)
    assert torch.equal(raw.argmax(1), calibrated.argmax(1))
    for version, probabilities in (("raw", raw), ("temperature_scaled", calibrated)):
        calibration_rows.append(capability_report(f"{source}:{version}", probabilities, data[1]))
calibration_comparison = pd.DataFrame(calibration_rows)
temperature_hash_after_site_c = canonical_hash(temperature_policy_fragment)
assert temperature_hash_before_site_c == temperature_hash_after_site_c
calibration_comparison


In [ ]:
classwise_rows = []
for source, data in (("Site B", site_b_dev), ("Site C", site_c_report)):
    outputs = ensemble_outputs(data[0])
    calibrated = probabilities_at_temperature(outputs["mean_logits"], SELECTED_TEMPERATURE)
    for class_id, class_name in enumerate(CLASS_NAMES):
        mask = data[1].eq(class_id)
        ece, _ = expected_calibration_error(calibrated[mask], data[1][mask], bins=5)
        classwise_rows.append({"source": source, "class": class_name, "rows": int(mask.sum()), "ece": ece})
classwise_calibration = pd.DataFrame(classwise_rows)
classwise_calibration


## 12. Embedding-distance, Mahalanobis, and energy OOD scores

Class centers and covariance use Site A training embeddings only. A diagonal regularizer prevents singular inversion. Every score is converted so larger means more OOD.


In [ ]:
train_outputs = ensemble_outputs(site_a_train[0])
train_embeddings = train_outputs["embedding"]
centroids = torch.stack([train_embeddings[site_a_train[1] == class_id].mean(0) for class_id in range(len(CLASS_NAMES))])
centered = train_embeddings - centroids[site_a_train[1]]
covariance = centered.T @ centered / max(1, len(centered) - 1)
REGULARIZATION = 1e-3
inverse_covariance = torch.linalg.inv(covariance + REGULARIZATION * torch.eye(covariance.shape[0]))


def normalize_ood_score(values: torch.Tensor, spec: ScoreSpec) -> torch.Tensor:
    if spec.normalization == "identity":
        normalized = values
    elif spec.normalization == "one_minus":
        normalized = 1 - values
    else:
        raise ValueError(f"unsupported_normalization:{spec.normalization}")
    return normalized


def ood_scores(images: torch.Tensor) -> pd.DataFrame:
    outputs = ensemble_outputs(images)
    probabilities = outputs["probabilities"]
    embedding = outputs["embedding"]
    deltas = embedding[:, None, :] - centroids[None, :, :]
    centroid_distance = torch.linalg.vector_norm(deltas, dim=2).min(1).values
    mahalanobis_squared = torch.einsum("nkd,df,nkf->nk", deltas, inverse_covariance, deltas)
    mahalanobis = mahalanobis_squared.clamp_min(0).sqrt().min(1).values
    energy = -torch.logsumexp(outputs["mean_logits"], dim=1)
    entropy = -(probabilities * probabilities.clamp_min(1e-9).log()).sum(1)
    raw_scores = {
        "msp": probabilities.max(1).values,
        "entropy": entropy,
        "energy": energy,
        "centroid_distance": centroid_distance,
        "mahalanobis": mahalanobis,
    }
    return pd.DataFrame({
        spec.normalized_name: normalize_ood_score(raw_scores[name], spec).numpy()
        for name, spec in SCORE_SPECS.items()
    })


score_orientation = {spec.normalized_name: "higher_is_more_ood" for spec in SCORE_SPECS.values()}
assert set(score_orientation.values()) == {"higher_is_more_ood"}
assert torch.allclose(normalize_ood_score(torch.tensor([0.9]), SCORE_SPECS["msp"]), torch.tensor([0.1]))


## 13. OOD metrics and a Site-B-only threshold

OOD is the positive class. AUROC/AUPR measure ranking. FPR@95 TPR names an operating region. A threshold chosen for 95% development OOD recall also exposes the ID false-reject rate.


In [ ]:
def fpr_at_target_tpr(labels: np.ndarray, scores: np.ndarray, target_tpr: float = 0.95) -> float:
    fpr, tpr, _ = roc_curve(labels, scores)
    eligible = np.where(tpr >= target_tpr)[0]
    return float(fpr[eligible[0]]) if len(eligible) else 1.0


id_dev_images = torch.cat([
    site_b_dev[0],
    corrupt(site_b_dev[0], "blur", 2, SEED + 83),
    corrupt(site_b_dev[0], "occlusion", 2, SEED + 84),
])
id_dev_scores = ood_scores(id_dev_images)
ood_dev_scores = pd.concat([ood_scores(near_ood_dev), ood_scores(far_ood_dev)], ignore_index=True)
ood_metric_rows = []
for score_name in id_dev_scores.columns:
    labels = np.concatenate([np.zeros(len(id_dev_scores)), np.ones(len(ood_dev_scores))])
    scores = np.concatenate([id_dev_scores[score_name], ood_dev_scores[score_name]])
    threshold = float(np.quantile(ood_dev_scores[score_name], 0.05, method="lower"))
    ood_metric_rows.append({
        "score": score_name,
        "positive_class": "OOD",
        "orientation": "higher_is_more_ood",
        "auroc": roc_auc_score(labels, scores),
        "aupr": average_precision_score(labels, scores),
        "fpr_at_95_tpr": fpr_at_target_tpr(labels, scores),
        "site_b_threshold_for_95pct_ood_recall": threshold,
        "id_false_reject_rate": float((id_dev_scores[score_name] >= threshold).mean()),
        "ood_recall": float((ood_dev_scores[score_name] >= threshold).mean()),
    })
ood_metrics = pd.DataFrame(ood_metric_rows).sort_values(["auroc", "id_false_reject_rate"], ascending=[False, True])
SELECTED_OOD_SCORE = str(ood_metrics.iloc[0]["score"])
SELECTED_OOD_THRESHOLD = float(ood_metrics.iloc[0]["site_b_threshold_for_95pct_ood_recall"])

# A deliberately invalid operating point: with higher-is-more-OOD scores,
# -infinity flags every example. Recall alone therefore cannot select policy.
ALL_REJECT_OOD_THRESHOLD = float("-inf")
all_reject_operating_point = {
    "threshold": ALL_REJECT_OOD_THRESHOLD,
    "ood_recall": float((ood_dev_scores[SELECTED_OOD_SCORE] >= ALL_REJECT_OOD_THRESHOLD).mean()),
    "id_false_reject_rate": float((id_dev_scores[SELECTED_OOD_SCORE] >= ALL_REJECT_OOD_THRESHOLD).mean()),
    "operationally_valid": False,
}
assert all_reject_operating_point["ood_recall"] == 1.0
assert all_reject_operating_point["id_false_reject_rate"] == 1.0
ood_metrics, all_reject_operating_point


### Why one scalar uncertainty score is insufficient

The same diagnostics answer different questions. This controlled comparison keeps labels only for evaluation: familiar ambiguity blends known classes, clean unsupported inputs move a known pattern outside its development support, and corrupted unsupported inputs add occlusion. Entropy and disagreement describe prediction geometry; embedding distance and the selected OOD score describe support; task error still requires labels.


In [ ]:
diagnostic_slices = (
    ("A · ambiguous but familiar", ambiguous_images, ambiguous_labels),
    ("B · clean but unsupported", unsupported_clean_images, unsupported_labels),
    ("C · corrupted and unsupported", corrupted_unsupported_images, unsupported_labels),
)
diagnostic_rows = []
for slice_name, images, labels in diagnostic_slices:
    outputs = ensemble_outputs(images)
    predictive = predictive_scores(outputs)
    support = ood_scores(images)
    diagnostic_rows.append({
        "slice": slice_name,
        "mean_entropy": float(predictive["entropy"].mean()),
        "mean_ensemble_disagreement": float(predictive["vote_disagreement"].mean()),
        "mean_embedding_distance": float(support["centroid_distance"].mean()),
        "selected_ood_score": SELECTED_OOD_SCORE,
        "mean_selected_ood_score": float(support[SELECTED_OOD_SCORE].mean()),
        "task_error_rate": float(outputs["probabilities"].argmax(1).ne(labels).float().mean()),
    })
uncertainty_failure_slices = pd.DataFrame(diagnostic_rows)
assert list(uncertainty_failure_slices["slice"].str[0]) == ["A", "B", "C"]
uncertainty_failure_slices


## 14. OOD is not error detection

The table crosses support status with task correctness. Separate error-detection AUROC uses entropy on labelled stressed-ID examples, not OOD labels.

![In-distribution predictions can be wrong and OOD predictions can be correct.](assets/ood-vs-error.svg)


In [ ]:
id_stress_images = torch.cat([site_b_dev[0], corrupt(site_b_dev[0], "occlusion", 4, SEED + 90), corrupt(site_b_dev[0], "blur", 5, SEED + 91)])
id_stress_labels = site_b_dev[1].repeat(3)
id_stress_outputs = ensemble_outputs(id_stress_images)
id_stress_prediction = id_stress_outputs["probabilities"].argmax(1)
error_labels = id_stress_prediction.ne(id_stress_labels).numpy().astype(int)
error_scores = predictive_scores(id_stress_outputs)["entropy"].numpy()
error_detection_auroc = roc_auc_score(error_labels, error_scores) if len(np.unique(error_labels)) == 2 else float("nan")

ood_predictions = ensemble_outputs(near_ood_report)["probabilities"].argmax(1)
# A synthetic semantic oracle assigns some novel examples to the nearest known
# appearance only to demonstrate the logical OOD/correctness cross-product.
ood_proxy_labels = ood_predictions.clone()
ood_proxy_labels[::4] = (ood_proxy_labels[::4] + 1) % len(CLASS_NAMES)

ood_vs_error = pd.DataFrame([
    {"support": "ID", "correctness": "correct", "count": int((~torch.tensor(error_labels, dtype=torch.bool)).sum())},
    {"support": "ID", "correctness": "wrong", "count": int(torch.tensor(error_labels, dtype=torch.bool).sum())},
    {"support": "OOD", "correctness": "correct under teaching proxy", "count": int(ood_predictions.eq(ood_proxy_labels).sum())},
    {"support": "OOD", "correctness": "wrong under teaching proxy", "count": int(ood_predictions.ne(ood_proxy_labels).sum())},
])
assert (ood_vs_error["count"] > 0).all()
{"error_detection_positive_class": "prediction error", "error_detection_auroc": error_detection_auroc}, ood_vs_error


## 15. Split-conformal prediction sets

Site B supplies nonconformity scores (1-p_y). The finite-sample corrected quantile targets 90% marginal coverage. Site C measures what happened under shift; it does not tune the quantile.


In [ ]:
CONFORMAL_ALPHA = 0.10
site_b_calibrated = probabilities_at_temperature(site_b_outputs["mean_logits"], SELECTED_TEMPERATURE)
nonconformity = 1 - site_b_calibrated[torch.arange(len(site_b_dev[1])), site_b_dev[1]]
quantile_level = min(1.0, math.ceil((len(nonconformity) + 1) * (1 - CONFORMAL_ALPHA)) / len(nonconformity))
CONFORMAL_QUANTILE = float(np.quantile(nonconformity.numpy(), quantile_level, method="higher"))


def conformal_report(source: str, data: tuple[torch.Tensor, torch.Tensor]) -> dict[str, float | str]:
    outputs = ensemble_outputs(data[0])
    probabilities = probabilities_at_temperature(outputs["mean_logits"], SELECTED_TEMPERATURE)
    prediction_set = probabilities >= (1 - CONFORMAL_QUANTILE)
    covered = prediction_set[torch.arange(len(data[1])), data[1]]
    sizes = prediction_set.sum(1)
    return {
        "source": source,
        "nominal_marginal_coverage": 1 - CONFORMAL_ALPHA,
        "observed_coverage": float(covered.float().mean()),
        "mean_set_size": float(sizes.float().mean()),
        "empty_set_rate": float(sizes.eq(0).float().mean()),
        "full_set_rate": float(sizes.eq(len(CLASS_NAMES)).float().mean()),
        "assumption": "exchangeable calibration/test examples; not guaranteed after arbitrary shift",
    }


conformal_results = pd.DataFrame([conformal_report("Site B development", site_b_dev), conformal_report("Site C reporting only", site_c_report)])
conformal_results


## 16. Risk–coverage and cost-sensitive threshold selection

Coverage and conditional error are reported together. Selective risk is mathematically undefined when no examples are accepted, so the notebook stores `NaN` rather than a misleading zero. Site B selects an operating point only after enforcing a minimum-coverage contract.


In [ ]:
def selective_point(probabilities: torch.Tensor, labels: torch.Tensor, threshold: float) -> dict[str, float | int]:
    confidence = probabilities.max(1).values
    correct = probabilities.argmax(1).eq(labels)
    accepted = confidence >= threshold
    accepted_count = int(accepted.sum())
    coverage = accepted_count / len(labels)
    selective_risk = float((~correct[accepted]).float().mean()) if accepted_count else float("nan")
    return {
        "threshold": float(threshold),
        "accepted_count": accepted_count,
        "coverage": float(coverage),
        "selective_risk": selective_risk,
    }


def risk_coverage_curve(probabilities: torch.Tensor, labels: torch.Tensor) -> pd.DataFrame:
    thresholds = torch.cat([torch.linspace(0.34, 0.99, 40), torch.tensor([1.01])])
    return pd.DataFrame([selective_point(probabilities, labels, float(threshold)) for threshold in thresholds])


known_probabilities = torch.tensor([[0.9, 0.1]])
known_answer_selective_risk = pd.DataFrame([
    {"case": "all accepted", **selective_point(known_probabilities, torch.tensor([0]), 0.0)},
    {"case": "none accepted", **selective_point(known_probabilities, torch.tensor([0]), 1.0)},
    {"case": "one wrong accepted", **selective_point(known_probabilities, torch.tensor([1]), 0.0)},
    {"case": "one correct accepted", **selective_point(known_probabilities, torch.tensor([0]), 0.0)},
])
known = known_answer_selective_risk.set_index("case")
assert known.loc["all accepted", "coverage"] == 1.0
assert known.loc["none accepted", "coverage"] == 0.0 and np.isnan(known.loc["none accepted", "selective_risk"])
assert known.loc["one wrong accepted", "selective_risk"] == 1.0
assert known.loc["one correct accepted", "selective_risk"] == 0.0


risk_dev_images = torch.cat([
    site_b_dev[0],
    corrupt(site_b_dev[0], "blur", 4, SEED + 92),
    corrupt(site_b_dev[0], "occlusion", 4, SEED + 93),
])
risk_dev_labels = site_b_dev[1].repeat(3)
risk_dev_outputs = ensemble_outputs(risk_dev_images)
risk_dev_probabilities = probabilities_at_temperature(risk_dev_outputs["mean_logits"], SELECTED_TEMPERATURE)
risk_coverage = risk_coverage_curve(risk_dev_probabilities, risk_dev_labels)
DEMONSTRATION_COSTS = {"wrong_accept": 12.0, "false_reject": 1.5, "human_review": 2.0, "reobserve": 0.4, "unresolved": 4.0}
MINIMUM_REQUIRED_COVERAGE = 0.55
risk_coverage["expected_cost_proxy"] = (
    risk_coverage["coverage"] * risk_coverage["selective_risk"] * DEMONSTRATION_COSTS["wrong_accept"]
    + (1 - risk_coverage["coverage"]) * DEMONSTRATION_COSTS["false_reject"]
)
empty_acceptance = risk_coverage.loc[risk_coverage["accepted_count"] == 0].iloc[0]
assert np.isnan(empty_acceptance["selective_risk"])
assert np.isnan(empty_acceptance["expected_cost_proxy"])
eligible_policy = risk_coverage.loc[
    (risk_coverage["coverage"] >= MINIMUM_REQUIRED_COVERAGE)
    & risk_coverage["selective_risk"].notna()
]
assert not eligible_policy.empty
selected_row = eligible_policy.loc[eligible_policy["expected_cost_proxy"].idxmin()]
assert selected_row["coverage"] >= MINIMUM_REQUIRED_COVERAGE
ACCEPT_CONFIDENCE_MIN = float(selected_row["threshold"])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(risk_coverage["coverage"], risk_coverage["selective_risk"], marker=".")
ax.scatter([selected_row["coverage"]], [selected_row["selective_risk"]], color="#F59E42", label="Site B policy")
ax.set(xlabel="coverage", ylabel="selective risk", title="Risk–coverage curve")
ax.legend()
plt.show()
known_answer_selective_risk, selected_row.to_dict()


## 17. Freeze the complete policy before Site C

The hash binds temperature, OOD score/threshold, confidence threshold, conformal quantile, retry budgets, freshness, and demonstration costs. Site C cannot change any field.


In [ ]:
policy_payload = {
    "version": "advanced07-reliability-policy-v1",
    "selected_on": "Site B development only",
    "temperature": SELECTED_TEMPERATURE,
    "ood_score_name": SELECTED_OOD_SCORE,
    "ood_threshold": SELECTED_OOD_THRESHOLD,
    "accept_confidence_min": ACCEPT_CONFIDENCE_MIN,
    "minimum_required_coverage": MINIMUM_REQUIRED_COVERAGE,
    "conformal_alpha": CONFORMAL_ALPHA,
    "conformal_quantile": CONFORMAL_QUANTILE,
    "max_reobservations": 1,
    "max_alternate_attempts": 1,
    "freshness_max_s": 2.0,
    "costs": DEMONSTRATION_COSTS,
}
FROZEN_POLICY_HASH = canonical_hash(policy_payload)
POLICY = ReliabilityPolicy(**policy_payload, digest=FROZEN_POLICY_HASH)
policy_hash_before_site_c = canonical_hash(asdict(POLICY))

# Reporting-only Site C evidence uses the already frozen policy.
site_c_outputs = ensemble_outputs(site_c_report[0])
site_c_probabilities = probabilities_at_temperature(site_c_outputs["mean_logits"], POLICY.temperature)
site_c_ood = ood_scores(site_c_report[0])[POLICY.ood_score_name].to_numpy()
site_c_policy_report = {
    **capability_report("Site C reporting only", site_c_probabilities, site_c_report[1]),
    "ood_flag_rate": float((site_c_ood >= POLICY.ood_threshold).mean()),
    "accept_rate": float(((site_c_probabilities.max(1).values.numpy() >= POLICY.accept_confidence_min) & (site_c_ood < POLICY.ood_threshold)).mean()),
}
policy_hash_after_site_c = canonical_hash(asdict(POLICY))
assert policy_hash_before_site_c == policy_hash_after_site_c
site_c_policy_report


## 18. Sensor validation, missing evidence, and fail-closed policy

Stale, missing, and contradictory inputs are application failures, not model uncertainty. A tri-state evidence table distinguishes `PASS`, `FAIL`, and `MISSING`; only complete passing evidence may be accepted.


In [ ]:
EvidenceState = Literal["PASS", "FAIL", "MISSING"]


def validate_sensor(observation: SensorObservation, now_s: float) -> dict[str, EvidenceState]:
    return {
        "image_present": "PASS" if observation.image_present else "FAIL",
        "freshness": "PASS" if now_s - observation.timestamp_s <= POLICY.freshness_max_s else "FAIL",
        "modality_agreement": "PASS" if observation.modality_agreement == "agree" else ("FAIL" if observation.modality_agreement == "contradict" else "MISSING"),
    }


def application_risk_gate(confidence: float | None, ood_score: float | None, sensor_checks: dict[str, EvidenceState]) -> tuple[EvidenceState, str]:
    if any(state == "FAIL" for state in sensor_checks.values()):
        return "FAIL", "sensor_contract_failed"
    if any(state == "MISSING" for state in sensor_checks.values()) or confidence is None or ood_score is None:
        return "MISSING", "required_evidence_missing"
    if confidence < POLICY.accept_confidence_min or ood_score >= POLICY.ood_threshold:
        return "FAIL", "risk_threshold_triggered"
    return "PASS", "complete_evidence_passed"


gate_assertions = pd.DataFrame([
    {"case": "complete evidence", "result": application_risk_gate(0.99, POLICY.ood_threshold - 1.0, {"image_present": "PASS", "freshness": "PASS", "modality_agreement": "PASS"})[0]},
    {"case": "confidence missing", "result": application_risk_gate(None, POLICY.ood_threshold - 1.0, {"image_present": "PASS", "freshness": "PASS", "modality_agreement": "PASS"})[0]},
    {"case": "modality unknown", "result": application_risk_gate(0.99, POLICY.ood_threshold - 1.0, {"image_present": "PASS", "freshness": "PASS", "modality_agreement": "MISSING"})[0]},
    {"case": "stale sensor", "result": application_risk_gate(0.99, POLICY.ood_threshold - 1.0, {"image_present": "PASS", "freshness": "FAIL", "modality_agreement": "PASS"})[0]},
])
assert gate_assertions.set_index("case").loc["complete evidence", "result"] == "PASS"
assert (gate_assertions.query("case != 'complete evidence'")["result"] != "PASS").all()
gate_assertions


## 19. Bounded re-observation, alternate method, and verification

The alternate method is nearest centroid in embedding space, not a second call to the same head. Re-observation and alternate attempts are each bounded to one. `bounded_recovery_policy()` has no label or verifier access: it may only emit a candidate or terminal non-success state. A separate finalizer consumes an evaluation-only receipt from the synthetic oracle. The fallback can never self-certify with its own support score.

![Recovery is bounded and ends in a typed, application-owned state.](assets/recovery-state-machine.svg)


In [ ]:
@torch.no_grad()
def primary_evidence(image: torch.Tensor) -> tuple[int, float, float]:
    outputs = ensemble_outputs(image.unsqueeze(0))
    probabilities = probabilities_at_temperature(outputs["mean_logits"], POLICY.temperature)[0]
    score = float(ood_scores(image.unsqueeze(0))[POLICY.ood_score_name].iloc[0])
    return int(probabilities.argmax()), float(probabilities.max()), score


@torch.no_grad()
def alternate_centroid(image: torch.Tensor) -> tuple[int, float]:
    embedding = ensemble_outputs(image.unsqueeze(0))["embedding"][0]
    distances = torch.linalg.vector_norm(embedding[None, :] - centroids, dim=1)
    prediction = int(distances.argmin())
    support = float(torch.exp(-distances.min()))
    return prediction, support


def candidate_result(candidate: RecoveryCandidate | None) -> dict[str, Any] | None:
    if candidate is None:
        return None
    return {
        "attempt_id": candidate.attempt_id,
        "method": candidate.method,
        "prediction": candidate.prediction,
        "decision_support": candidate.decision_support,
    }


ALTERNATE_MINIMUM_SUPPORT = 0.20


def alternate_candidate_proposal(candidate: RecoveryCandidate, attempts: int) -> RecoveryProposal:
    if candidate.decision_support >= ALTERNATE_MINIMUM_SUPPORT:
        return RecoveryProposal(candidate.operation_id, "verification_required", "alternate_candidate_requires_independent_verification", attempts, candidate)
    return RecoveryProposal(candidate.operation_id, "human_review", "alternate_candidate_below_support_contract", attempts, candidate)


def bounded_recovery_policy(case: dict[str, Any], now_s: float = 100.0) -> RecoveryProposal:
    # Produce a bounded proposal without labels or verification access.
    operation_id = case["operation_id"]
    observation = case["observation"]
    attempts = 0
    checks = validate_sensor(observation, now_s)

    if not observation.image_present and case.get("reobservation") is None:
        return RecoveryProposal(operation_id, "blocked_invalid_input", "image_missing_no_reobservation", attempts, None)

    image = case.get("primary_image")
    primary_available = case.get("primary_available", True)
    prediction = confidence = ood_score = None
    if image is not None and primary_available:
        prediction, confidence, ood_score = primary_evidence(image)
    state, reason = application_risk_gate(confidence, ood_score, checks)
    if state == "PASS":
        candidate = RecoveryCandidate(operation_id, f"{operation_id}:primary:0", "primary", int(prediction), float(confidence))
        return RecoveryProposal(operation_id, "accepted", "primary_complete_evidence", attempts, candidate)

    if case.get("reobservation") is not None and attempts < POLICY.max_reobservations:
        attempts += 1
        replacement_observation, replacement_image = case["reobservation"]
        replacement_checks = validate_sensor(replacement_observation, now_s)
        prediction, confidence, ood_score = primary_evidence(replacement_image)
        replacement_state, _ = application_risk_gate(confidence, ood_score, replacement_checks)
        if replacement_state == "PASS":
            candidate = RecoveryCandidate(operation_id, f"{operation_id}:reobservation:{attempts}", "reobservation", prediction, confidence)
            return RecoveryProposal(operation_id, "verification_required", "fresh_reobservation_requires_independent_verification", attempts, candidate)

    if image is not None and attempts < POLICY.max_reobservations + POLICY.max_alternate_attempts:
        attempts += 1
        alternate_prediction, alternate_support = alternate_centroid(image)
        candidate = RecoveryCandidate(operation_id, f"{operation_id}:alternate:{attempts}", "alternate_centroid", alternate_prediction, alternate_support)
        return alternate_candidate_proposal(candidate, attempts)

    terminal = "human_review" if case.get("review_available", True) else "unresolved"
    return RecoveryProposal(operation_id, terminal, reason, attempts, None)


def finalize_with_independent_verification(
    proposal: RecoveryProposal,
    verifier: Callable[[RecoveryCandidate], VerificationReceipt],
) -> RiskDecision:
    # Finalize outside the recovery policy; only this layer can access a verifier.
    receipt = None
    if proposal.candidate is not None and proposal.terminal_state in {"accepted", "verification_required"}:
        receipt = verifier(proposal.candidate)
        assert receipt.attempt_id == proposal.candidate.attempt_id

    if proposal.terminal_state == "verification_required":
        terminal_state = "verified_recovery" if receipt and receipt.verified_success else "human_review"
        reason = "independent_recovery_verification_passed" if terminal_state == "verified_recovery" else "independent_recovery_verification_failed"
    else:
        terminal_state = proposal.terminal_state
        reason = proposal.reason

    return RiskDecision(
        operation_id=proposal.operation_id,
        terminal_state=terminal_state,
        reason=reason,
        attempts=proposal.attempts,
        recovery_attempt=None if proposal.candidate is None or proposal.candidate.method == "primary" else proposal.candidate.method,
        candidate_result=candidate_result(proposal.candidate),
        verification_source=receipt.verification_source if receipt else None,
        verified_success=receipt.verified_success if receipt else False,
    )


def run_bounded_recovery(
    case: dict[str, Any],
    verifier: Callable[[RecoveryCandidate], VerificationReceipt],
    now_s: float = 100.0,
) -> RiskDecision:
    proposal = bounded_recovery_policy(case, now_s)
    return finalize_with_independent_verification(proposal, verifier)


## 20. Recovery cases include transient, semantic, multimodal, and system failures

Cases inject clean operation, severe blur with a fresh second observation, stale input, missing image, contradictory modality, novel OOD, and unavailable primary model. Labels live in a separate evaluation-oracle mapping, not in policy cases. No fallback failure silently releases the original result.


In [ ]:
def observation(case_id: str, timestamp_s: float, present: bool = True, agreement: str = "agree") -> SensorObservation:
    return SensorObservation(case_id, "Site C", timestamp_s, present, agreement)


clean_image = site_c_report[0][0]
true_label = int(site_c_report[1][0])
blurred_image = corrupt(clean_image.unsqueeze(0), "blur", 5, SEED + 120)[0]
cases = [
    {"operation_id": "clean", "observation": observation("clean", 99.5), "primary_image": clean_image},
    {"operation_id": "blur_reobserve", "observation": observation("blur", 99.5), "primary_image": blurred_image, "reobservation": (observation("blur_retry", 100.0), clean_image)},
    {"operation_id": "stale_reobserve", "observation": observation("stale", 90.0), "primary_image": clean_image, "reobservation": (observation("stale_retry", 100.0), clean_image)},
    {"operation_id": "missing", "observation": observation("missing", 99.8, present=False), "primary_image": None, "review_available": False},
    {"operation_id": "contradictory", "observation": observation("contradict", 99.8, agreement="contradict"), "primary_image": clean_image},
    {"operation_id": "near_ood", "observation": observation("near", 99.8), "primary_image": near_ood_report[0]},
    {"operation_id": "primary_unavailable", "observation": observation("unavailable", 99.8), "primary_image": clean_image, "primary_available": False},
]

EVALUATION_ORACLE_LABELS = {
    "clean": true_label,
    "blur_reobserve": true_label,
    "stale_reobserve": true_label,
    "missing": true_label,
    "contradictory": true_label,
    "near_ood": -1,
    "primary_unavailable": true_label,
}


def synthetic_evaluation_oracle(candidate: RecoveryCandidate) -> VerificationReceipt:
    return VerificationReceipt(
        attempt_id=candidate.attempt_id,
        verification_source="synthetic_evaluation_oracle",
        verified_success=candidate.prediction == EVALUATION_ORACLE_LABELS[candidate.operation_id],
    )


assert all("true_label" not in case for case in cases)
assert "verifier" not in bounded_recovery_policy.__code__.co_varnames
assert "EVALUATION_ORACLE_LABELS" not in bounded_recovery_policy.__code__.co_names
recovery_decisions = pd.DataFrame([asdict(run_bounded_recovery(case, synthetic_evaluation_oracle)) for case in cases])
assert (recovery_decisions["authorization"] == "none").all()
assert not ((recovery_decisions["terminal_state"] == "verified_recovery") & (~recovery_decisions["verified_success"])).any()
assert not ((recovery_decisions["terminal_state"] == "verified_recovery") & recovery_decisions["verification_source"].isna()).any()
assert recovery_decisions.loc[recovery_decisions["operation_id"] == "missing", "terminal_state"].iloc[0] == "blocked_invalid_input"

# Known-answer invariants: support can admit a candidate, but only the separate
# oracle receipt can verify it. Low-support correctness cannot bypass policy.
adversarial_oracle = {"wrong": 0, "uncertain_correct": 1, "correct": 2}
def adversarial_verifier(candidate: RecoveryCandidate) -> VerificationReceipt:
    return VerificationReceipt(candidate.attempt_id, "synthetic_evaluation_oracle", candidate.prediction == adversarial_oracle[candidate.operation_id])

confident_wrong = RecoveryCandidate("wrong", "wrong:alternate:1", "alternate_centroid", 1, 0.99)
uncertain_correct = RecoveryCandidate("uncertain_correct", "uncertain:alternate:1", "alternate_centroid", 1, 0.05)
confident_correct = RecoveryCandidate("correct", "correct:alternate:1", "alternate_centroid", 2, 0.99)
recovery_invariant_tests = pd.DataFrame([
    asdict(finalize_with_independent_verification(alternate_candidate_proposal(confident_wrong, 1), adversarial_verifier)),
    asdict(finalize_with_independent_verification(alternate_candidate_proposal(uncertain_correct, 1), adversarial_verifier)),
    asdict(finalize_with_independent_verification(alternate_candidate_proposal(confident_correct, 1), adversarial_verifier)),
])
invariants = recovery_invariant_tests.set_index("operation_id")
assert invariants.loc["wrong", "terminal_state"] == "human_review" and not invariants.loc["wrong", "verified_success"]
assert invariants.loc["uncertain_correct", "terminal_state"] == "human_review" and pd.isna(invariants.loc["uncertain_correct", "verification_source"])
assert invariants.loc["correct", "terminal_state"] == "verified_recovery" and invariants.loc["correct", "verified_success"]
recovery_decisions, recovery_invariant_tests


## 21. Always-answer, abstain-only, and bounded-recovery policies

The comparison separates accepted work, attempted recovery, verified success, false success claims, review, unresolved work, and invalid-input blocks. Safety and utility remain visible rather than collapsed into one score.


In [ ]:
def evaluate_policy(name: str) -> dict[str, float | int | str]:
    outcomes = []
    for case in cases:
        if name == "always_answer":
            if case.get("primary_image") is None or not case.get("primary_available", True):
                outcomes.append(("unresolved", False, 0))
            else:
                prediction, _, _ = primary_evidence(case["primary_image"])
                outcomes.append(("accepted", prediction == EVALUATION_ORACLE_LABELS[case["operation_id"]], 0))
        elif name == "abstain_only":
            checks = validate_sensor(case["observation"], 100.0)
            if case.get("primary_image") is None or not case.get("primary_available", True):
                outcomes.append(("human_review", False, 0))
            else:
                prediction, confidence, score = primary_evidence(case["primary_image"])
                state, _ = application_risk_gate(confidence, score, checks)
                outcomes.append(("accepted" if state == "PASS" else "human_review", state == "PASS" and prediction == EVALUATION_ORACLE_LABELS[case["operation_id"]], 0))
        else:
            decision = run_bounded_recovery(case, synthetic_evaluation_oracle)
            outcomes.append((decision.terminal_state, decision.verified_success, decision.attempts))
    verified = sum(int(item[1]) for item in outcomes)
    false_success = sum(int(item[0] in {"accepted", "verified_recovery"} and not item[1]) for item in outcomes)
    return {
        "policy": name,
        "cases": len(outcomes),
        "verified_successes": verified,
        "verified_success_rate": verified / len(outcomes),
        "false_success_claims": false_success,
        "human_review": sum(int(item[0] == "human_review") for item in outcomes),
        "unresolved_or_blocked": sum(int(item[0] in {"unresolved", "blocked_invalid_input"}) for item in outcomes),
        "total_recovery_attempts": sum(item[2] for item in outcomes),
    }


policy_comparison = pd.DataFrame([evaluate_policy(name) for name in ("always_answer", "abstain_only", "bounded_recovery")])
policy_comparison


## 22. Failure attribution and enterprise evidence

The final artifact keeps task, calibration, OOD, error-detection, conformal, selective, sensor, and recovery evidence separate. It records no hidden reasoning and grants no production authority.


In [ ]:
failure_attribution = pd.DataFrame([
    {"stage": "input", "signal": "missing/stale/contradictory modality", "response": "block, re-observe, or review"},
    {"stage": "capability", "signal": "corruption/source degradation", "response": "slice investigation and bounded policy"},
    {"stage": "calibration", "signal": "confidence/accuracy gap", "response": "recalibrate on development data; do not claim correction"},
    {"stage": "support", "signal": "OOD score over frozen threshold", "response": "abstain or governed fallback"},
    {"stage": "selective_policy", "signal": "risk or coverage outside contract", "response": "review threshold/cost on new development version"},
    {"stage": "recovery", "signal": "fallback attempted but unverified", "response": "human review or unresolved"},
    {"stage": "monitoring", "signal": "shift alert without labels", "response": "investigate; do not infer task loss"},
])


def json_records(frame: pd.DataFrame) -> list[dict[str, Any]]:
    # Use JSON null, never non-standard NaN/Infinity tokens, in durable evidence.
    return json.loads(frame.to_json(orient="records"))


evidence = {
    "course": "Advanced 07 — Robustness, Uncertainty & Failure Recovery",
    "evidence_version": "1",
    "environment": environment,
    "local_proxy": {"engine": "procedural_images_tiny_cnn_ensemble", "foundation_model": False, "production_result": False},
    "source_contracts": [asdict(item) for item in SOURCES],
    "metric_specs": {name: asdict(spec) for name, spec in METRIC_SPECS.items()},
    "score_specs": {name: asdict(spec) for name, spec in SCORE_SPECS.items()},
    "split_manifest": split_manifest.to_dict(orient="records"),
    "base_ensemble_hash": BASE_ENSEMBLE_HASH,
    "base_metrics": base_metrics.to_dict(orient="records"),
    "corruption_results": corruption_results.to_dict(orient="records"),
    "degradation_summary": degradation_summary.to_dict(orient="records"),
    "uncertainty_comparison": uncertainty_comparison.to_dict(orient="records"),
    "uncertainty_failure_slices": uncertainty_failure_slices.to_dict(orient="records"),
    "false_confidence_example": false_confidence_example,
    "calibration_comparison": calibration_comparison.to_dict(orient="records"),
    "classwise_calibration": classwise_calibration.to_dict(orient="records"),
    "ood_score_orientation": score_orientation,
    "ood_metrics": ood_metrics.to_dict(orient="records"),
    "all_reject_ood_operating_point": {**all_reject_operating_point, "threshold": "-infinity"},
    "error_detection": {"positive_class": "prediction error", "auroc": error_detection_auroc},
    "conformal_results": conformal_results.to_dict(orient="records"),
    "risk_coverage": json_records(risk_coverage),
    "known_answer_selective_risk": json_records(known_answer_selective_risk),
    "reliability_policy": asdict(POLICY),
    "site_c_reporting_only": site_c_policy_report,
    "policy_hash_unchanged_after_site_c": policy_hash_before_site_c == policy_hash_after_site_c,
    "gate_assertions": gate_assertions.to_dict(orient="records"),
    "recovery_decisions": json_records(recovery_decisions),
    "recovery_invariant_tests": json_records(recovery_invariant_tests),
    "policy_comparison": json_records(policy_comparison),
    "failure_attribution": failure_attribution.to_dict(orient="records"),
    "optional_tool_manifests": OPTIONAL_TOOL_MANIFESTS,
    "unresolved_production_assumptions": [
        "real source representativeness, label quality, consent, privacy, and retention",
        "task-specific near/far/operational OOD definitions",
        "conditional calibration and conformal behavior under nonstationarity",
        "target-hardware ensemble latency, availability, and correlated failures",
        "delayed outcome joins, human-review capacity, incident response, and recovery drills",
    ],
    "authorization": "none",
}

evidence_path = ARTIFACT_DIR / "robustness_uncertainty_recovery_evidence.json"
decision_path = ARTIFACT_DIR / "robustness_uncertainty_recovery_decisions.csv"
evidence_path.write_text(json.dumps(evidence, indent=2, default=str, allow_nan=False) + "\n", encoding="utf-8")
recovery_decisions.to_csv(decision_path, index=False)

loaded = json.loads(evidence_path.read_text(encoding="utf-8"))
assert loaded["authorization"] == "none"
assert loaded["local_proxy"]["foundation_model"] is False
assert loaded["policy_hash_unchanged_after_site_c"] is True
evidence_path, decision_path


## 23. Production upgrade map and exercises

| Teaching path | Production upgrade |
| --- | --- |
| procedural images | licensed, source/time/site-isolated visual data |
| immediate labels | delayed-outcome joins and label-quality governance |
| tiny local ensemble | pinned artifacts, target-hardware cost and parity |
| synthetic OOD | task-specific validated support and incident slices |
| in-memory policy | authenticated versioned service and separation of duties |
| simulation oracle | delayed ground truth, expert review, or verified postcondition |
| local JSON/CSV | signed registry, retention, audit, rollback and recovery drills |

Exercises:

1. Add motion blur and preserve the corruption/severity contract.
2. Compare diagonal and full regularized Mahalanobis distance.
3. Add adaptive prediction sets and report classwise coverage.
4. Inject fallback unavailability and prove the original risky result is not released.
5. Replace one-point costs with a sensitivity analysis while keeping Site C reporting-only.
6. Design monitoring that distinguishes shift alerts, confirmed capability loss, and recovery incidents.

### Final checkpoint

You should now be able to explain why confidence is not uncertainty, calibration is not correction, OOD detection is not error detection, low selective risk is meaningless without coverage, and a recovery attempt is not a verified recovery.
